In [1]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "neurocnl",
#     "snntorch==0.9.4",
#     "torch==2.13.0",
#     "tonic",
# ]
# ///

# NeuroMorphic Pipeline — Snntorch Sim

Generated 2026-07-23 14:56 UTC.

**Architecture:** defined in the Architecture tab (CNL spec below).
**Pipeline config:** edit `config` in the next cell to change training parameters.

In [2]:
# ── Pipeline configuration ───────────────────────────────────────────
# Workspace settings used when this notebook was generated.

config = {
    "dataset":   "tonic_shd",
    "framework": "snntorch_sim",
}

print('Config loaded:', config)

Config loaded: {'dataset': 'tonic_shd', 'framework': 'snntorch_sim'}


In [3]:
import json as _json


def _nmtk_emit(
    epoch: int, total: int, loss: float, accuracy: float, layer_rates: dict,
    phase: str = "train",
) -> None:
    print(
        _json.dumps(
            {
                "__nmtk_progress__": True,
                "epoch": epoch,
                "total_epochs": total,
                "loss": loss,
                "accuracy": accuracy,
                "layer_spike_rates": layer_rates,
                "phase": phase,
            }
        ),
        flush=True,
    )


In [4]:
import torch
from torch.utils.data import DataLoader
try:
    import tonic
    train_ds = tonic.datasets.SHD(save_to='data/', train=True)
    test_ds  = tonic.datasets.SHD(save_to='data/', train=False)
except ImportError:
    raise ImportError('pip install tonic')
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0)
print(f'SHD: {len(train_loader.dataset)} train / {len(test_loader.dataset)} test samples')

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


130864128it [00:16, 8007276.51it/s]                                


Extracting data/SHD/shd_train.h5.zip to data/SHD


38141952it [00:00, 89936643.57it/s]                               


Extracting data/SHD/shd_test.h5.zip to data/SHD
SHD: 8156 train / 2264 test samples


## Architecture

Network compiled from CNL spec via NIR.

In [5]:
# CNL spec — auto-generated from the Architecture canvas tab.
# To change the network, edit the Architecture tab and regenerate.
cnl_spec = '''
Define a network named shd_digit_classifier with timestep 0.001.
# Network with 1 input, 5 hidden nodes, 1 output.
# flow: input → cochlea → w_cochlea_hidden → hidden → w_hidden_classes → classes → output

# Layers:
Define an input port named input with shape (700,).
Define a LIF neuron named cochlea with time constant 0.02, resistance 1.0, leak voltage 0.0, and firing threshold 0.8.
Define a linear transformation named w_cochlea_hidden with weight matrix shape (128, 700), annotated with metadata seed equal to 42, and annotated with metadata weight_init equal to "xavier".
Define a LIF neuron named hidden with time constant 0.02, resistance 1.0, leak voltage 0.0, and firing threshold 0.6.
Define a linear transformation named w_hidden_classes with weight matrix shape (20, 128), annotated with metadata seed equal to 43, and annotated with metadata weight_init equal to "xavier".
Define a LIF neuron named classes with time constant 0.03, resistance 1.0, leak voltage 0.0, and firing threshold 0.7.
Define an output port named output with shape (20,).

# Connections:
input connects to cochlea.
cochlea connects to w_cochlea_hidden.
w_cochlea_hidden connects to hidden.
hidden connects to w_hidden_classes.
w_hidden_classes connects to classes.
classes connects to output.

'''

from neurocnl.compile import compile_to_nir

graph = compile_to_nir(cnl_spec)
print(f'Network: {len(graph.nodes)} nodes, {len(graph.edges)} edges')

Network: 7 nodes, 6 edges


In [6]:
import torch
import numpy as np
torch.manual_seed(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

"""snnTorch network — auto-generated from NIR graph."""

import torch
import torch.nn as nn
import snntorch as snn
import numpy as np

from snntorch import surrogate
spike_grad = surrogate.fast_sigmoid(slope=25.0)

_w = np.load('weights_snntorch_sim_f914b9d1.npz')  # weights file saved alongside this notebook

_expected_weight_keys = ['w_cochlea_hidden_weight', 'w_hidden_classes_weight']
_missing_weight_keys = [k for k in _expected_weight_keys if k not in _w.files]
if _missing_weight_keys:
    raise RuntimeError(
        f"'weights_snntorch_sim_f914b9d1.npz' is missing {_missing_weight_keys} — this weights "
        "file does not match the current network. Regenerate the notebook from "
        "the Architecture tab so its weights file matches this architecture."
    )

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # LIF population: 'cochlea'  (1 neurons)
        self.cochlea = snn.Leaky(beta=0.950000, threshold=16.0000, reset_mechanism='zero', init_hidden=True, reset_delay=False, spike_grad=spike_grad)
        # Linear layer: 'w_cochlea_hidden'  shape (128, 700)
        self.w_cochlea_hidden = nn.Linear(700, 128)
        self.w_cochlea_hidden.weight.data = torch.from_numpy(_w['w_cochlea_hidden_weight'].copy())
        self.w_cochlea_hidden.bias = None
        # LIF population: 'hidden'  (1 neurons)
        self.hidden = snn.Leaky(beta=0.950000, threshold=12.0000, reset_mechanism='zero', init_hidden=True, reset_delay=False, spike_grad=spike_grad)
        # Linear layer: 'w_hidden_classes'  shape (20, 128)
        self.w_hidden_classes = nn.Linear(128, 20)
        self.w_hidden_classes.weight.data = torch.from_numpy(_w['w_hidden_classes_weight'].copy())
        self.w_hidden_classes.bias = None
        # LIF population: 'classes'  (1 neurons)
        self.classes = snn.Leaky(beta=0.966667, threshold=21.0000, reset_mechanism='zero', init_hidden=True, reset_delay=False, spike_grad=spike_grad)

    def forward(self, x):
        # initialise hidden states
        self.cochlea.init_leaky()
        self.hidden.init_leaky()
        self.classes.init_leaky()
        # x: (T,B,C,H,W) time-first from tonic, or (B,C,H,W) for a single frame
        if x.dim() == 4:
            x = x.unsqueeze(0)  # (B,C,H,W) → (1,B,C,H,W)
        elif x.dim() == 3 and x.shape[0] <= x.shape[1]:
            x = x.swapaxes(0, 1)  # (B,T,N) → (T,B,N) synthetic spike batches
        elif x.dim() == 2:
            x = x.unsqueeze(0).expand(globals().get('num_steps', 1), -1, -1)  # (B,F) static features → (T,B,F), repeated every step (rate coding)
        _x_seq = x
        spk_rec = []
        mem_rec = []
        for t in range(_x_seq.shape[0]):
            x = _x_seq[t]
            spk_cochlea = self.cochlea(x)
            x = spk_cochlea
            x = self.w_cochlea_hidden(x)
            spk_hidden = self.hidden(x)
            x = spk_hidden
            x = self.w_hidden_classes(x)
            spk_classes = self.classes(x)
            mem_rec.append(getattr(self.classes, 'mem', spk_classes).clone())
            x = spk_classes
            spk_rec.append(x)
        mem_out = torch.stack(mem_rec, dim=0) if mem_rec else x
        return torch.stack(spk_rec, dim=0), mem_out  # (T, batch, out), membrane trace or last activation


net = Net().float()  # ponytail: npz weights load as float64; cast to match DataLoader float32 input
print(f'Net: {sum(p.numel() for p in net.parameters())} parameters')

Net: 92160 parameters


## Train

In [7]:
import tonic
import tonic.transforms as transforms
from torch.utils.data import DataLoader

_sensor_size = tonic.datasets.SHD.sensor_size
_frame_tf = transforms.ToFrame(sensor_size=_sensor_size, time_window=1000)
_pad_collate = tonic.collation.PadTensors(batch_first=False)  # pad variable-length event frames

train_ds = tonic.datasets.SHD(save_to='./data', train=True,  transform=_frame_tf)
test_ds  = tonic.datasets.SHD(save_to='./data', train=False, transform=_frame_tf)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True,  num_workers=0, collate_fn=_pad_collate)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False, num_workers=0, collate_fn=_pad_collate)
num_steps = 25
optimizer = torch.optim.Adam(
    net.parameters(), lr=0.001, weight_decay=0.0,
    betas=(0.9, 0.999)
)
_vl_every_n_epochs = 1
_vl_save_best_checkpoint = True
_vl_checkpoint_metric = 'val_accuracy'
_vl_checkpoint_mode = 'max'
_vl_best = float('-inf')
val_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
net.train()
for epoch in range(50):
    _epoch_loss_sum = 0.0; _epoch_batches = 0
    for batch_idx, (data, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        # Reset network membrane potentials
        for layer in net.modules():
            if hasattr(layer, 'reset_mem'):
                layer.reset_mem()
        spk_out, mem_out = net(data)
        import snntorch.functional as SF
        loss_fn = SF.mse_count_loss(correct_rate=0.8, incorrect_rate=0.2)
        loss_val = loss_fn(spk_out, targets)
        loss_val.backward()
        print(f'  loss: {loss_val.item():.4f}')
        optimizer.step()
        _epoch_loss_sum += loss_val.item(); _epoch_batches += 1
    avg_loss = _epoch_loss_sum / _epoch_batches if _epoch_batches else 0.0
    _nmtk_emit(
        epoch=epoch + 1,
        total=50,
        loss=avg_loss,
        accuracy=None,
        layer_rates={'output': float(spk_out.float().mean().item())},
        phase='train',
    )
    if (epoch + 1) % _vl_every_n_epochs == 0:
        net.eval()
        _vl_correct = 0
        _vl_total = 0
        _vl_loss_sum = 0.0
        _vl_batches = 0
        with torch.no_grad():
            for _vl_data, _vl_targets in val_loader:
                _vl_spk_out, _vl_mem_out = net(_vl_data)
                _vl_loss_sum += loss_fn(_vl_spk_out, _vl_targets).item(); _vl_batches += 1
                _vl_correct += (_vl_spk_out.sum(0).argmax(1) == _vl_targets).sum().item()
                _vl_total += _vl_targets.size(0)
        val_loss = _vl_loss_sum / _vl_batches if _vl_batches else 0.0
        val_accuracy = _vl_correct / _vl_total if _vl_total else 0.0
        _nmtk_emit(
            epoch=epoch + 1,
            total=50,
            loss=(val_loss if val_loss is not None else 0.0),
            accuracy=val_accuracy,
            layer_rates={},
            phase='val',
        )
        _vl_metric_value = (
            val_accuracy if _vl_checkpoint_metric == 'val_accuracy'
            else (val_loss if val_loss is not None else val_accuracy)
        )
        _vl_improved = (
            _vl_metric_value >= _vl_best if _vl_checkpoint_mode == 'max'
            else _vl_metric_value <= _vl_best
        )
        if _vl_improved:
            _vl_best = _vl_metric_value
            if _vl_save_best_checkpoint:
                torch.save(net.state_dict(), 'best_model.pt')
        net.train()

/usr/local/lib/python3.11/site-packages/tonic/datasets/hsd.py:24: RuntimeWarning: overflow encountered in cast
  file["spikes/times"][index] * 1e6,
/usr/local/lib/python3.11/site-packages/tonic/datasets/hsd.py:24: RuntimeWarning: invalid value encountered in multiply
  file["spikes/times"][index] * 1e6,
/usr/local/lib/python3.11/site-packages/tonic/io.py:31: RuntimeWarning: invalid value encountered in cast
  struct_arr[name] = arg
/usr/local/lib/python3.11/site-packages/torch/nn/modules/loss.py:630: UserWarning: Using a target size (torch.Size([32, 20])) that is different to the input size (torch.Size([1, 32, 1, 20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.0000
  loss: 0.

/usr/local/lib/python3.11/site-packages/torch/nn/modules/loss.py:630: UserWarning: Using a target size (torch.Size([28, 20])) that is different to the input size (torch.Size([1, 28, 1, 20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.11/site-packages/torch/nn/modules/loss.py:630: UserWarning: Using a target size (torch.Size([32, 20])) that is different to the input size (torch.Size([32, 1, 1, 20])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (20) must match the size of tensor b (32) at non-singleton dimension 2

## Evaluate

In [ ]:
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)
try:
    net.load_state_dict(torch.load('best_model.pt', weights_only=True))
    print('Loaded best checkpoint from best_model.pt')
except FileNotFoundError:
    print("best_model.pt not found — evaluating with current in-memory weights "
          "(enable 'Save Best Checkpoint' on the Validation Loop node and train first).")
correct = 0; total = 0
net.eval()
with torch.no_grad():
    for batch_idx, (data, targets) in enumerate(test_loader):
        # Reset network membrane potentials
        for layer in net.modules():
            if hasattr(layer, 'reset_mem'):
                layer.reset_mem()
        
        spk_out, mem_out = net(data)
        
        correct += (spk_out.sum(0).argmax(1) == targets).sum().item()
        total   += targets.size(0)
        _nmtk_emit(
            epoch=batch_idx + 1,
            total=len(test_loader),
            loss=0.0,
            accuracy=(correct / total if total else None),
            layer_rates={'output': float(spk_out.float().mean().item())},
            phase='eval',
        )
print(f'Accuracy (top-1): {correct/total:.2%}' if total else 'Accuracy (top-1): n/a')
_nmtk_emit(
    epoch=1,
    total=1,
    loss=0.0,
    accuracy=(correct / total if total else None),
    layer_rates={'output': float(spk_out.float().mean().item())},
    phase='eval',
)

In [ ]:
# ── Download as Python script ──────────────────────────────────
# Run this cell to download the notebook as a .py script.
import subprocess
subprocess.run(['jupyter', 'nbconvert', '--to', 'script',
                '__file__'], check=False)
print('Conversion triggered — check the file listing.')